# Three-way LR verification and LaTeX table benchmarks

This SageMath notebook evaluates three implementations of the same Littlewood–Richardson multiplicities:

1. **Original:** unmodified `hoyt.ipynb`.
2. **Targeted:** cached and subpartition-restricted `hoyt_large_scale_regression.ipynb`.
3. **Redesigned:** bilinear kernel and sparse coproduct code from `hoyt_optimized.ipynb`.

It tests numerical equality, benchmarks **one-table generation** and separates the **new degree band** from the full cumulative `le_generateTables(n,n)` workload. **No full LaTeX tables are displayed.** The original table writer stops at the first empty row; the redesigned writer checks every degree. For that reason, identical numerical coefficients do not guarantee byte-for-byte equivalent LaTeX tables.

Default settings use a very small sample for degrees **6 and 7**. Increase samples after confirming Sage performance. Timings are measured on your machine and are not prefilled.

In [2]:
import builtins, csv, io, json, random, re, statistics, time
from collections import defaultdict
from contextlib import redirect_stdout
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import HTML, display
from functools import lru_cache
import sage.libs.lrcalc.lrcalc as lrcalc

# All three notebook files must be beside this benchmarking notebook.
ORIGINAL_PATH = Path("hoyt.ipynb")
TARGETED_PATH = Path("hoyt_large_scale_regression.ipynb")
REDESIGN_PATH = Path("hoyt_optimized.ipynb")
for p in (ORIGINAL_PATH, TARGETED_PATH, REDESIGN_PATH):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required notebook: {p}")

# A stable way to execute selected Python-definition cells without running examples.
def notebook_cells(path):
    return json.loads(path.read_text(encoding="utf-8"))["cells"]

def load_definitions(path, indices, source_overrides=None):
    cells = notebook_cells(path)
    ns = {"__name__": "hoyt_benchmark_" + path.stem, "__builtins__": __builtins__, "re": re}
    for index in indices:
        src = "".join(cells[index]["source"])
        if source_overrides and index in source_overrides:
            src = source_overrides[index](src)
        with redirect_stdout(io.StringIO()):
            exec(compile(src, f"{path.name}:cell_{index}", "exec"), ns)
    return ns

def trim_original_partitions(src):
    return src.split("\np = (3, 1)")[0]
def trim_original_subset(src):
    return src.split("\nsubset_partitions((3,1))")[0]

# Load exactly the original functions, including its unmodified old table writer.
original = load_definitions(ORIGINAL_PATH,
    [0,1,3,4,8,11,14,18,20,23],
    {3:trim_original_partitions, 4:trim_original_subset})
# The targeted notebook preserves the original table writer but optimizes LR internals.
targeted = load_definitions(TARGETED_PATH,
    [0,1,3,4,8,11,14,18,20,23])
# Redesigned notebook stores code in these positions.
# Cell 7 defines partitions_of(), required by generate_tables().
# Skip the assertions and example tests that follow those definitions.
def trim_redesigned_verification(src):
    return src.split("\nassert triple_lr(", 1)[0]

redesigned = load_definitions(REDESIGN_PATH, [1,3,5,7,9],
    {7: trim_redesigned_verification})

for name, ns, required in [
    ("original", original, ("CalcSoc", "MakeTable_tex", "le_generateTables")),
    ("targeted", targeted, ("CalcSoc", "MakeTable_tex", "le_generateTables")),
    ("redesigned", redesigned, ("row_data", "table_tex", "kernel", "partitions_of", "generate_tables"))]:
    absent=[f for f in required if f not in ns]
    if absent: raise RuntimeError(f"{name} missing: {absent}")
print("Loaded all three independent implementations.")


FileNotFoundError: Missing required notebook: hoyt_large_scale_regression.ipynb

In [15]:
# Configuration: integer values explicitly converted to native Python types for Sage compatibility.
SEED = int(20261028)
DEGREES = (6, 9)                    # Can extend to (6,7,8), etc.
SAMPLES_PER_DEGREE_PAIR = int(10)  # Sample pairs (lambda, mu) at each equal degree.
MAX_ORIGINAL_DEGREE = int(6)      # Restrict very slow original runs; set to 7 or 8 voluntarily.
MAX_TARGETED_DEGREE = int(9)
CHECKS_PER_TABLE = int(20)         # Coefficient spot checks per table per k (see below).
SAVE_LATEX_FILES = False          # Timed output is written to temp files anyway.
OUTPUT_PREFIX = "hoyt_three_way"
rng = random.Random(int(SEED))

# Helpers for reproducible selection, with no Sage random-seed incompatibility.
def partitions_of(n):
    return [tuple(p) for p in original["generate_partitions"](int(n))]

def candidate_pairs(n):
    parts=partitions_of(n)
    pairs=[(a,b) for a in parts for b in parts]
    rng.shuffle(pairs)
    return pairs[:min(int(SAMPLES_PER_DEGREE_PAIR),len(pairs))]

def clear_caches(ns):
    for name in ("_subset_partitions_cached", "_lrcoeff_cached", "lrcoef3_cached",
                 "lr", "coproduct", "triple_lr", "coproduct_by_right", "kernel", "row_data"):
        f=ns.get(name)
        if f is not None and hasattr(f,"cache_clear"):
            f.cache_clear()

# Count cumulative tables without running le_generateTables itself.
def partition_count(n):
    return len(partitions_of(n))
def degree_band_tables(n):
    # New tables introduced by moving cumulative bound from n-1 to n.
    return partition_count(n)*builtins.sum(partition_count(j) for j in range(n+1))*2 - partition_count(n)**2
def cumulative_tables(n):
    return builtins.sum(partition_count(j) for j in range(n+1))**2

print("Configuration:", {"degrees":DEGREES,"samples_per_degree":SAMPLES_PER_DEGREE_PAIR,
      "max_original_degree":MAX_ORIGINAL_DEGREE,"max_targeted_degree":MAX_TARGETED_DEGREE})


Configuration: {'degrees': (6, 9), 'samples_per_degree': 10, 'max_original_degree': 6, 'max_targeted_degree': 9}


## Numerical verification

Compare each sampled nonzero row entry **and sampled zero entries** for every degree \(0\le k\le\min(|\lambda|,|\mu|)\). The redesigned result is calculated by `row_data`, while the two reference results are calculated by `CalcSoc`. When a reference implementation has been disabled at a higher degree, comparisons against that implementation are explicitly marked *not run*—never counted as passing.

In [16]:
verification=[]
selected={n:candidate_pairs(n) for n in DEGREES}
for n in DEGREES:
    for lam,mu in selected[n]:
        for k in range(n+1):
            res=redesigned["row_data"](int(k),lam,mu)
            lp=[tuple(p) for p in targeted["subset_partitions"](lam) if builtins.sum(p)==n-k]
            mp=[tuple(p) for p in targeted["subset_partitions"](mu) if builtins.sum(p)==n-k]
            possible=[(a,b) for a in lp for b in mp]
            rng.shuffle(possible)
            chosen=possible[:int(CHECKS_PER_TABLE)]
            # Explicitly include nonzero results, which are mathematically informative.
            for key in list(res)[:min(3,int(CHECKS_PER_TABLE))]:
                if key not in chosen: chosen.append(key)
            for a,b in chosen:
                fast=int(res.get((a,b),0))
                mid=None; old=None
                if n<=MAX_TARGETED_DEGREE:
                    with redirect_stdout(io.StringIO()):
                        mid=int(targeted["CalcSoc"](int(k),lam,a,mu,b))
                if n<=MAX_ORIGINAL_DEGREE:
                    with redirect_stdout(io.StringIO()):
                        old=int(original["CalcSoc"](int(k),lam,a,mu,b))
                verification.append({"degree":n,"lambda":str(lam),"mu":str(mu),"k":k,
                   "lambda_prime":str(a),"mu_prime":str(b),"original":old,
                   "targeted":mid,"redesigned":fast,
                   "original_agrees":old==fast if old is not None else "NOT RUN",
                   "targeted_agrees":mid==fast if mid is not None else "NOT RUN"})
    print(f"Verified sampled coefficients at degree {n}: {len(verification)} cumulative checks")

mismatches=[r for r in verification if r["original_agrees"] is False or r["targeted_agrees"] is False]
checks_old=builtins.sum(isinstance(r["original_agrees"],bool) for r in verification)
checks_mid=builtins.sum(isinstance(r["targeted_agrees"],bool) for r in verification)
print(f"Checks: {len(verification)} redesigned, {checks_mid} targeted, {checks_old} original; mismatches: {len(mismatches)}")
with open(OUTPUT_PREFIX+"_mismatches.csv","w",newline="",encoding="utf-8") as f:
    cols=list(verification[0]) if verification else ["degree","lambda","mu","k","lambda_prime","mu_prime","original","targeted","redesigned","original_agrees","targeted_agrees"]
    w=csv.DictWriter(f,fieldnames=cols); w.writeheader(); w.writerows(mismatches)
if mismatches:
    for r in mismatches[:10]: print("MISMATCH",r)


Verified sampled coefficients at degree 6: 222 cumulative checks
Verified sampled coefficients at degree 9: 905 cumulative checks
Checks: 905 redesigned, 905 targeted, 222 original; mismatches: 0


## Table-generation timing (degree \(>5\))

Each sample generates a complete table for a **specific partition pair**, including the original version's disk write. This is the workload of one `MakeTable_tex` call. The redesigned version writes its generated text as well. Cold timings clear memoized coefficient data first; warm timings repeat the same table with the warmed caches. The output size in bytes is also recorded.

**Note:** Sage's `le_generateTables(n,n)` performs a cumulative sweep and appends repeatedly to one file. Running it unchanged for large `n` can produce extremely large output. The cell below estimates batch scaling from sampled *individual-table* timings but does not claim the extrapolation is an actual measured batch runtime.

In [17]:
bench=[]
def measure_table(name,ns,lam,mu,path):
    clear_caches(ns)
    if path.exists(): path.unlink()
    def invoke():
        if name=="redesigned":
            tex=ns["table_tex"](lam,mu)
            path.write_text(tex,encoding="utf-8")
        else:
            with redirect_stdout(io.StringIO()):
                ns["MakeTable_tex"](lam,mu,str(path))
    start=time.perf_counter(); invoke(); cold=float(time.perf_counter()-start)
    bytes_count=path.stat().st_size
    start=time.perf_counter(); invoke(); warm=float(time.perf_counter()-start)
    return cold,warm,bytes_count

with TemporaryDirectory(prefix="hoyt_bench_") as tmp:
    temp=Path(tmp)
    for n in DEGREES:
        for j,(lam,mu) in enumerate(selected[n]):
            for name,ns,limit in (("original",original,MAX_ORIGINAL_DEGREE),
                                  ("targeted",targeted,MAX_TARGETED_DEGREE),
                                  ("redesigned",redesigned,max(DEGREES))):
                if n>limit: continue
                outfile=temp / f"{name}_{n}_{j}.tex"
                cold,warm,bytecount=measure_table(name,ns,lam,mu,outfile)
                bench.append({"degree":int(n),"lambda":str(lam),"mu":str(mu),
                    "algorithm":name,"cold_seconds":float(cold),
                    "warm_seconds":float(warm),"output_bytes":int(bytecount)})
                print(f"n={n} {name:10s} cold={cold:.3f}s warm={warm:.3f}s bytes={bytecount}")
                if SAVE_LATEX_FILES:
                    persistent=Path("hoyt_benchmark_tex");persistent.mkdir(exist_ok=True)
                    (persistent/outfile.name).write_bytes(outfile.read_bytes())

with open(OUTPUT_PREFIX+"_timings.csv","w",newline="",encoding="utf-8") as f:
    cols=["degree","lambda","mu","algorithm","cold_seconds","warm_seconds","output_bytes"]
    w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(bench)
print("Saved",OUTPUT_PREFIX+"_timings.csv")


n=6 original   cold=0.020s warm=0.020s bytes=1077
n=6 targeted   cold=0.004s warm=0.002s bytes=1070
n=6 redesigned cold=0.001s warm=0.000s bytes=926
n=6 original   cold=0.012s warm=0.012s bytes=500
n=6 targeted   cold=0.001s warm=0.001s bytes=496
n=6 redesigned cold=0.001s warm=0.000s bytes=431
n=6 original   cold=0.018s warm=0.018s bytes=988
n=6 targeted   cold=0.004s warm=0.002s bytes=982
n=6 redesigned cold=0.002s warm=0.000s bytes=836
n=6 original   cold=0.019s warm=0.019s bytes=996
n=6 targeted   cold=0.004s warm=0.002s bytes=989
n=6 redesigned cold=0.001s warm=0.000s bytes=865
n=6 original   cold=0.019s warm=0.020s bytes=1255
n=6 targeted   cold=0.004s warm=0.002s bytes=1248
n=6 redesigned cold=0.002s warm=0.000s bytes=1065
n=6 original   cold=0.018s warm=0.019s bytes=1131
n=6 targeted   cold=0.004s warm=0.002s bytes=1124
n=6 redesigned cold=0.001s warm=0.000s bytes=972
n=6 original   cold=0.013s warm=0.013s bytes=743
n=6 targeted   cold=0.003s warm=0.001s bytes=738
n=6 redesigne

In [18]:
# Compact monochrome summary; no pandas, no matplotlib, no huge LaTeX output.
from html import escape
lookup=defaultdict(list)
for r in bench: lookup[(int(r["degree"]),r["algorithm"])].append(r)

def fmt(x):return f"{float(x):.4f}"
rows=[]
for n in DEGREES:
    for alg in ("original","targeted","redesigned"):
        rs=lookup[(n,alg)]
        if not rs:
            rows.append((n,alg,"not run","—","—","—","—","—"))
        else:
            cold=statistics.median(float(x["cold_seconds"]) for x in rs)
            warm=statistics.median(float(x["warm_seconds"]) for x in rs)
            rows.append((n,alg,str(len(rs)),fmt(cold),fmt(warm),
                        str(int(partition_count(n)**2)),str(int(degree_band_tables(n))),str(int(cumulative_tables(n)))))

style="""<style>.hoyt-bench{font-family:Arial,sans-serif;color:#111;max-width:980px}
.hoyt-bench h2{font-size:21px;margin-bottom:8px}.hoyt-bench .meta{color:#555;font-size:13px}
.hoyt-bench table{border-collapse:collapse;width:100%;font-size:13px;margin:14px 0}
.hoyt-bench th{background:#111;color:white;text-align:right;padding:10px}
.hoyt-bench td{padding:9px 10px;border-bottom:1px solid #ddd;text-align:right}
.hoyt-bench th:nth-child(2),.hoyt-bench td:nth-child(2){text-align:left}
.hoyt-bench tbody tr:nth-child(even){background:#f4f4f4}
.hoyt-bench .metric{border:1px solid #bbb;border-radius:6px;display:inline-block;padding:13px 18px;margin:8px 7px 8px 0}
.hoyt-bench .metric b{display:block;font-size:20px}</style>"""
html=style+"<div class='hoyt-bench'><h2>LR table-generation performance</h2>"
html+=f"<div class='metric'><b>{len(verification):,}</b>Numerical checks</div><div class='metric'><b>{len(mismatches):,}</b>Mismatches</div><div class='metric'><b>{len(bench):,}</b>Timed table runs</div>"
html+="<p class='meta'>Cold and warm timings are measured per single (λ, μ) table. New-band and cumulative counts describe the full le_generateTables workload, not measured batch times.</p>"
html+="<table><thead><tr>"+"".join(f"<th>{escape(h)}</th>" for h in ["Degree","Method","Samples","Cold median (s)","Warm median (s)","Pairs at (n,n)","New-band tables","Cumulative tables"])+"</tr></thead><tbody>"
for r in rows:
    html+="<tr>"+"".join("<td>"+escape(str(x))+"</td>" for x in r)+"</tr>"
html+="</tbody></table><p class='meta'>A degree-n row of le_generateTables(n,n) contains every (i,j) degree pair with 0 ≤ i,j ≤ n. "+"Original and targeted stop writing table rows at their first zero; redesigned checks all k. Output lengths may differ even when common multiplicities agree.</p></div>"
display(HTML(html))


Degree,Method,Samples,Cold median (s),Warm median (s),"Pairs at (n,n)",New-band tables,Cumulative tables
6,original,10,0.0186,0.0187,121,539,900
6,targeted,10,0.0041,0.0020,121,539,900
6,redesigned,10,0.0014,0.0001,121,539,900
9,original,not run,—,—,—,—,—
9,targeted,10,0.0299,0.0108,900,4920,9409
9,redesigned,10,0.0052,0.0003,900,4920,9409


## Optional: measure the exact cumulative `le_generateTables` call

Turn on `RUN_FULL_BATCH` to time the actual legacy `le_generateTables(n,n)` function, not a sampled proxy. Its output is isolated in a temporary directory so it does not overwrite your own LaTeX tables. The redesigned implementation's `generate_tables(n,n, filename)` has equivalent cumulative degree bounds, but its row-stopping semantics differ. This block can become **very expensive for n≥6**; the default is off and has no timeout inside SageMath.

In [4]:
RUN_FULL_BATCH = True
FULL_BATCH_DEGREES = (9,)   # Increase only after seeing the sampled results.
FULL_BATCH_METHODS = ["redesigned", "original"]  # Add "original" explicitly for full baseline.

full_batch_timings=[]
if RUN_FULL_BATCH:
    import os
    previous_cwd=Path.cwd()
    with TemporaryDirectory(prefix="hoyt_fullbatch_") as tmp:
        root=Path(tmp)
        (root/"work").mkdir()
        (root/"hoyt_tables").mkdir()
        try:
            os.chdir(root/"work")
            for n in FULL_BATCH_DEGREES:
                for method in FULL_BATCH_METHODS:
                    ns={"original":original,"targeted":targeted,"redesigned":redesigned}[method]
                    clear_caches(ns)
                    started=time.perf_counter()
                    if method == "redesigned":
                        destination=root/"hoyt_tables"/f"redesigned_{n}.tex"
                        with redirect_stdout(io.StringIO()):
                            ns["generate_tables"](int(n),int(n),str(destination))
                    else:
                        # Exact original function uses ../hoyt_tables/t_all.tex.
                        destination=root/"hoyt_tables"/"t_all.tex"
                        if destination.exists(): destination.unlink()
                        with redirect_stdout(io.StringIO()):
                            ns["le_generateTables"](int(n),int(n))
                    elapsed=float(time.perf_counter()-started)
                    full_batch_timings.append({"degree":int(n),"algorithm":method,
                        "elapsed_seconds":elapsed,"bytes_written":destination.stat().st_size,
                        "tables_expected":int(cumulative_tables(n))})
                    print(f"Full cumulative degree ≤ {n} | {method}: {elapsed:.2f}s, {destination.stat().st_size:,} bytes")
        finally:
            os.chdir(previous_cwd)
    with open(OUTPUT_PREFIX+"_full_batch.csv","w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["degree","algorithm","elapsed_seconds","bytes_written","tables_expected"])
        w.writeheader();w.writerows(full_batch_timings)
else:
    print("Exact cumulative batch benchmark skipped. Set RUN_FULL_BATCH=True to enable.")


NameError: name 'original' is not defined

### How to interpret performance

- **Cold**: first table after clearing memoization; includes LR computation, formatting, and one file write.
- **Warm**: immediate repeat with cached LR values; includes formatting and another write.
- **New-band table count**: number of additional tables introduced when raising `le_generateTables(n-1,n-1)` to `le_generateTables(n,n)`.
- **Cumulative table count**: full number of tables emitted by `le_generateTables(n,n)`; equals \((\sum_{i=0}^n p(i))^2\), where \(p(i)\) is the partition number.

The three versions **do not produce identical LaTeX text by design**, because the redesigned writer explicitly checks all possible rows, while the original `MakeTable_tex` stops at its first zero row. If a table differs, compare the complete underlying coefficient maps rather than only the LaTeX source. For large degrees, first benchmark degree 6 with two samples; then enable original degree 7 if desired.